# Multi-Seed Batch Evaluation & Semantic Re-ranking
This notebook evaluates the Visual Product Search Engine across multiple CLIP finetuning seeds.
- **A**: Vision-only CLIP (frozen, evaluated once)
- **B**: Frozen CLIP + BLIP-2 reranking (evaluated once)
- **C**: Fine-tuned CLIP + BLIP-2 reranking (evaluated per-seed)


In [ ]:
!pip install -q omegaconf timm fairscale iopath decord webdataset pycocotools pycocoevalcap
!pip install -q --no-dependencies salesforce-lavis
!pip install -q transformers==4.38.2 open_clip_torch pinecone pandas Pillow tqdm scikit-learn accelerate

In [ ]:
import os
import torch
import open_clip
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm
from pinecone import Pinecone
from lavis.models import load_model_and_preprocess
import numpy as np
import math

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

In [ ]:
# ==============================================================================
# CONFIGURATIONS
# ==============================================================================
PINECONE_API_KEY  = "YOUR_PINECONE_API_KEY"  # Replace with your key
INDEX_NAME        = "vr-clothing-gallery"

DATASET_ROOT      = "/kaggle/input/datasets/vinay1706/deepfashion-cropped"
QUERY_CSV         = f"{DATASET_ROOT}/query.csv"
GALLERY_CSV       = f"{DATASET_ROOT}/gallery.csv"
CAPTIONS_CSV      = f"{DATASET_ROOT}/blip2_captions_gallery.csv"
IMAGE_ROOT        = DATASET_ROOT

TOP_K_RETRIEVAL = 15
K_VALUES = [5, 10, 15]

# ── Multi-seed configuration ──────────────────────────────────────────────────
SEEDS = [42, 123]

# Frozen configs (seed-independent, evaluated once)
FROZEN_CONFIGS = [
    ("Config A (Vision-Only)", "frozen-alpha-1.0", False, False),
    ("Config B (Frozen CLIP, alpha=0.7)", "frozen-alpha-0.7", False, True),
    ("Config B (Frozen CLIP, alpha=0.5)", "frozen-alpha-0.5", False, True),
]

# Finetuned configs are built per-seed using namespace pattern: finetuned-alpha-{A}-seed{S}
FINETUNED_ALPHAS = [0.7, 0.5]


In [ ]:
# ==============================================================================
# LOAD DATA
# ==============================================================================
query_df = pd.read_csv(QUERY_CSV)
gallery_df = pd.read_csv(GALLERY_CSV)
captions_df = pd.read_csv(CAPTIONS_CSV)

caption_map = dict(zip(captions_df['image_name'], captions_df['blip2_caption']))
relevance_count = gallery_df.groupby('item_id').size().to_dict()

print(f"Loaded {len(query_df)} query images.")
print(f"Loaded {len(gallery_df)} gallery images.")


In [ ]:
# ==============================================================================
# LOAD MODELS
# ==============================================================================
print("Loading LAVIS BLIP-2 ITM Model...")
blip_model, vis_processors, txt_processors = load_model_and_preprocess(
    name="blip2_image_text_matching",
    model_type="pretrain",
    is_eval=True,
    device=device
)

class PatchedTokenizer:
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer
    def __call__(self, *args, **kwargs):
        kwargs["padding"] = True
        return self.tokenizer(*args, **kwargs)
    def __getattr__(self, name):
        return getattr(self.tokenizer, name)

blip_model.tokenizer = PatchedTokenizer(blip_model.tokenizer)

def load_clip(checkpoint_path=None):
    model, _, preprocess = open_clip.create_model_and_transforms('ViT-L-14', pretrained='openai')
    if checkpoint_path is not None:
        ckpt = torch.load(checkpoint_path, map_location=device)
        state_dict = ckpt.get("model_state", ckpt)
        state_dict = {k.replace("module.", ""): v for k, v in state_dict.items()}
        model.load_state_dict(state_dict, strict=False)
        print(f"  Loaded finetuned weights from {checkpoint_path}")
    return model.to(device).eval(), preprocess

print("Connecting to Pinecone...")
pc = Pinecone(api_key=PINECONE_API_KEY)
index = pc.Index(INDEX_NAME)


In [ ]:
def compute_itm_scores_batched(raw_image, candidate_captions):
    img = vis_processors["eval"](raw_image).unsqueeze(0).to(device)
    img_batch = img.repeat(len(candidate_captions), 1, 1, 1)
    txt_batch = [txt_processors["eval"](c) for c in candidate_captions]
    with torch.no_grad():
        itm_output = blip_model({"image": img_batch, "text_input": txt_batch}, match_head="itm")
        itm_scores = torch.nn.functional.softmax(itm_output, dim=1)[:, 1].cpu().tolist()
    return itm_scores

In [ ]:
def calculate_metrics(ranked_item_ids, ground_truth_id, k_values=[5, 10, 15]):
    metrics = {}
    for k in k_values:
        top_k_items = ranked_item_ids[:k]

        total_relevant = relevance_count.get(ground_truth_id, 1)
        relevant_in_top_k = sum(1 for item in top_k_items if item == ground_truth_id)
        metrics[f'Recall@{k}'] = relevant_in_top_k / total_relevant

        dcg = 0.0
        for i, item in enumerate(top_k_items):
            if item == ground_truth_id:
                dcg += 1.0 / math.log2(i + 2)
        n_ideal = min(total_relevant, k)
        idcg = sum(1.0 / math.log2(i + 2) for i in range(n_ideal))
        metrics[f'NDCG@{k}'] = dcg / idcg if idcg > 0 else 0.0

        ap = 0.0
        relevant_count_so_far = 0
        for i, item in enumerate(top_k_items):
            if item == ground_truth_id:
                relevant_count_so_far += 1
                precision_at_i = relevant_count_so_far / (i + 1)
                ap += precision_at_i
        metrics[f'mAP@{k}'] = ap / total_relevant

    return metrics

In [ ]:
def evaluate_configs(configs, eval_df):
    # Evaluate a list of (config_name, namespace, checkpoint_path_or_None, use_reranking) tuples.
    results = []
    for config_name, namespace, checkpoint_path, use_reranking in configs:
        print(f"\n{'='*60}\nEvaluating {config_name}\n{'='*60}")

        clip_model, clip_preprocess = load_clip(checkpoint_path)

        config_metrics = {f'Recall@{k}': [] for k in K_VALUES}
        config_metrics.update({f'NDCG@{k}': [] for k in K_VALUES})
        config_metrics.update({f'mAP@{k}': [] for k in K_VALUES})

        for idx, row in tqdm(eval_df.iterrows(), total=len(eval_df)):
            query_image_path = os.path.join(IMAGE_ROOT, row['image_name'])
            gt_item_id = row['item_id']

            try:
                raw_image = Image.open(query_image_path).convert('RGB')
                processed_image = clip_preprocess(raw_image).unsqueeze(0).to(device)
            except Exception:
                continue

            with torch.no_grad():
                with torch.amp.autocast('cuda'):
                    query_embedding = clip_model.encode_image(processed_image)
                    query_embedding = torch.nn.functional.normalize(query_embedding, dim=-1).cpu().numpy().tolist()[0]

            retrieval_response = index.query(
                vector=query_embedding,
                top_k=TOP_K_RETRIEVAL,
                namespace=namespace,
                include_metadata=True
            )
            candidates = retrieval_response['matches']

            if use_reranking:
                cand_images = [c['metadata']['image_name'] for c in candidates]
                cand_captions = [caption_map.get(img, "") for img in cand_images]
                itm_scores = compute_itm_scores_batched(raw_image, cand_captions)
                scored = sorted(zip(candidates, itm_scores), key=lambda x: x[1], reverse=True)
                candidates = [s[0] for s in scored]

            ranked_item_ids = [c['metadata']['item_id'] for c in candidates]
            q_metrics = calculate_metrics(ranked_item_ids, gt_item_id, K_VALUES)
            for k, v in q_metrics.items():
                config_metrics[k].append(v)

        print(f"\nResults for {config_name}:")
        summary_row = {"Config": config_name}
        for metric, values in config_metrics.items():
            mean_val = np.mean(values)
            summary_row[metric] = mean_val
            print(f"{metric}: {mean_val:.4f}")
        results.append(summary_row)

        del clip_model
        torch.cuda.empty_cache()

    return results

In [ ]:
# ==============================================================================
# MAIN EVALUATION
# ==============================================================================
eval_df = query_df  # Full query set

all_results = []

# ── Step 1: Evaluate frozen configs (once) ────────────────────────────────────
frozen_eval_configs = [
    (name, ns, None, rerank)
    for name, ns, _, rerank in FROZEN_CONFIGS
]
all_results.extend(evaluate_configs(frozen_eval_configs, eval_df))

# ── Step 2: Evaluate finetuned configs (per-seed) ────────────────────────────
for seed in SEEDS:
    checkpoint = f"{DATASET_ROOT}/clip_best_seed{seed}.pt"
    print(f"\n{'#'*60}")
    print(f"# SEED {seed}")
    print(f"{'#'*60}")

    if not os.path.exists(checkpoint):
        print(f"  WARNING: {checkpoint} not found — skipping seed {seed}")
        continue

    seed_configs = []
    for alpha in FINETUNED_ALPHAS:
        namespace = f"finetuned-alpha-{alpha}-seed{seed}"
        config_name = f"Config C (FT, alpha={alpha}, seed={seed})"
        seed_configs.append((config_name, namespace, checkpoint, True))

    all_results.extend(evaluate_configs(seed_configs, eval_df))

# ── Final Summary ─────────────────────────────────────────────────────────────
results_df = pd.DataFrame(all_results)
display(results_df)
results_df.to_csv("/kaggle/working/eval_results_multiseed.csv", index=False)
print("\nResults saved to /kaggle/working/eval_results_multiseed.csv")
